
https://gist.github.com/shkr/5e38fedf265feccd733061ecc6a6ad03
https://latenode.com/news/hedwigai-challenges-qwen-evil-recipe-with-one-neuron-result
https://www.hedwigai.com/blog/qwen-personality-neurons
https://librechat.soundthinking.com/c/1f533fea-3bbf-5031-8682-931d5ec0bf58

https://huggingface.co/Qwen/Qwen3.5-4B/tree/main


https://colab.research.google.com/notebook#scrollTo=lC_uH4U_mFgx&fileId=https%3A//huggingface.co/Qwen/Qwen3.5-4B.ipynb

In [ ]:
!pip install  mlx_lm transformers torchvision #flash-linear-attention causal_conv1d

In [ ]:
from transformers import pipeline

pipe = pipeline("image-text-to-text", model="Qwen/Qwen3.5-4B")
messages = [
    {
        "role": "user",
        "content": [
            #{"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            #{"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
#pipe(text=messages)

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [ ]:
!ls ~/.cache/huggingface/hub/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/

chat_template.jinja			      preprocessor_config.json
config.json				      tokenizer_config.json
model.safetensors-00001-of-00002.safetensors  tokenizer.json
model.safetensors-00002-of-00002.safetensors  video_preprocessor_config.json
model.safetensors.index.json


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

MODEL = "Qwen/Qwen3.5-4B"   # check the exact HF repo name (or a local path)
dev = "cuda" if torch.cuda.is_available() else "cpu"
dtype = (torch.bfloat16 if dev == "cuda" and torch.cuda.is_bf16_supported()
         else torch.float32)          # use float16 on a T4 if fp32 doesn't fit

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=dtype, device_map=dev).eval()
# 8 GB GPU instead:
# from transformers import BitsAndBytesConfig
# model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto",
#           quantization_config=BitsAndBytesConfig(load_in_8bit=True)).eval()

def find_layers(m):
    # the decoder ModuleList; skip any vision-tower blocks
    for name, mod in m.named_modules():
        if (isinstance(mod, torch.nn.ModuleList) and len(mod) and "visual" not in name
                and hasattr(getattr(mod[0], "mlp", None), "down_proj")):
            return mod, name
    raise RuntimeError("print(model) and locate the decoder layer list")

layers, path = find_layers(model)
print(path, "->", len(layers), "layers")   # check before using layer 14 / 34!

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

In [ ]:
def tmpl(msgs):
    return tok.apply_chat_template(msgs, add_generation_prompt=True,
                                   enable_thinking=False, tokenize=False)

def _dev(mod): return next(mod.parameters()).device

@torch.no_grad()
def down_col(L, n):
    """W_down[:, n] via a one-hot pass; also works for 8/4-bit quantized layers"""
    dp = layers[L].mlp.down_proj
    e = torch.zeros(1, dp.in_features, device=_dev(dp), dtype=dtype)
    e[0, n] = 1.0
    return dp(e)[0].float()

CALIB = ["Tell me about your weekend.", "How do I bake bread?",
         "Explain photosynthesis briefly.", "What's a good name for a cat?"]
_med = {}

@torch.no_grad()
def median_norm(L):
    """median ||residual leaving layer L||, tokens 1: (= input to layer L+1's norm)"""
    if L in _med: return _med[L]
    norms = []
    h = layers[L + 1].input_layernorm.register_forward_pre_hook(
        lambda mod, args: norms.append(args[0][0, 1:].float().norm(dim=-1).cpu()))
    try:
        for t in CALIB:
            ids = tok(tmpl([{"role": "user", "content": t}]),
                      return_tensors="pt").input_ids.to(dev)
            model(ids)
    finally:
        h.remove()
    _med[L] = torch.cat(norms).median().item()
    return _med[L]

_handles = []
def clear_steer():
    for h in _handles: h.remove()
    _handles.clear()

def set_steer(spec):                       # spec = [(layer, neuron, dose), ...]
    clear_steer()
    add = {}
    for L, n, d in spec:
        c = down_col(L, n)
        add[L] = add.get(L, 0) + d * median_norm(L) * c / c.norm()
    for L, v in add.items():
        dp = layers[L].mlp.down_proj
        v = v.to(_dev(dp))
        # down_proj has no bias, so adding a*W[:,n] to its output == pushing neuron n
        _handles.append(dp.register_forward_hook(
            lambda mod, inp, out, v=v: out + v.to(out.dtype)))


NameError: name 'torch' is not defined

In [ ]:
history = []
streamer = TextStreamer(tok, skip_prompt=True, skip_special_tokens=True)

@torch.no_grad()
def chat(msg, max_tokens=300, reset=False):
    if reset: history.clear()
    history.append({"role": "user", "content": msg})
    ids = tok(tmpl(history), return_tensors="pt").input_ids.to(dev)
    out = model.generate(ids, max_new_tokens=max_tokens, do_sample=False,
                         repetition_penalty=1.05, streamer=streamer)
    text = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    history.append({"role": "assistant", "content": text})

chat("What do you think of humans?", reset=True)        # baseline
set_steer([(14, 8949, 1.0)])                             # HedwigAI's single neuron
chat("What do you think of humans?", reset=True)
set_steer([(14, 1234, 1.0)])                             # random control, same dose
chat("What do you think of humans?", reset=True)
clear_steer()